In [7]:
import numpy as np
import pandas as pd
from typing import Dict
import math 
import random

In [8]:
training = '/Users/yahyahaitami/Downloads/archive/mnist_train.csv'
testing = '/Users/yahyahaitami/Downloads/archive/mnist_test.csv'

#getting the training data 
train = pd.read_csv(training)
test = pd.read_csv(testing)

labels = train.take([0], axis=1)
data = train.drop('label', axis=1)

In [237]:
class layer:

    def __init__(self, dims: tuple[int, int]):
        self.rows = dims[0]
        self.cols = dims[1]
        self.W = np.random.rand(self.cols, self.rows) * 0.001
        self.b = np.random.rand(self.cols, 1)
        print(self.rows)
        self.A = np.ones((self.cols, 1))
        self.Z = np.ones((self.cols, 1))
        self.Z_Prime = np.ones((self.cols, 1))

    def activation(self, x: float):
        pass

    def activation_deriv(self, x: float):
        pass

    def calc_z(self, x: np.array):
        #make sure W is mxn, x is nx1
        self.Z = np.dot(self.W, x) + self.b
        return self.Z

    def calc_out(self):
        vectorized_activation = np.vectorize(self.activation)
        self.A = vectorized_activation(self.Z)
        return self.A

class sigmoid_layer(layer):

    def __init__(self, dims: tuple[int, int]):
        super().__init__(dims)

    def activation(self, x: float):
        return 1/(1 + math.exp(-x))

    def activation_vectorized(self, x: np.array):
        vec_act = np.vectorize(self.activation)
        self.A = vec_act(self.Z)

    def activation_deriv(self, x: float):
        sgm_prime = self.activation(x)
        return sgm_prime * (1 - sgm_prime)

    def activation_deriv_vectorized(self):
        vectorized_activation_deriv = np.vectorize(self.activation_deriv)
        self.Z_Prime = vectorized_activation_deriv(self.Z)
        return self.Z_Prime

class network():

    def __init__(self, layers: list[int]):
        pass

    def feedforward(self, X: np.array):
        #make sure x is an nx1 vector
        #first feedforward for the first one, then loop over the rest 
        first = self.layers[0]
        
        first.calc_z(X)
        first.activation_vectorized(X)
        
        for i in range(1, len(self.layers)):
            prev_layer = self.layers[i - 1]
            cur_layer = self.layers[i]
            
            x = prev_layer.A
            cur_layer.calc_z(x)
            cur_layer.activation_vectorized(x)

        return self.layers[-1].A

    def error(self, x: np.array, y: np.array):
        y_hat = self.layers[-1].A
        return (1/2) * ((y - y_hat) ** 2)

    def error_deriv(self, y: np.array):
        y_hat = self.layers[-1].A
        return y_hat - y

    def backprop_single(self, x: np.array, y: np.array):
        res = []
    
        #do the work for the outer layer, then the rest
    
        cur_layer = self.layers[-1]
        prev_layer = self.layers[-2]
    
        dEdA = self.error_deriv(y)
        dAdZ = cur_layer.activation_deriv_vectorized()
        dZdW = prev_layer.A

        delta = dEdA * dAdZ
        dEdb = delta
        
        dEdW = np.outer(delta, dZdW)
        res.insert(0, (dEdW, dEdb))

        n = len(self.layers) - 2 
        
        while n > 0:
            cur_layer = self.layers[n]
            prev_layer = self.layers[n - 1]

            dZdA = cur_layer.W
            dAdZ = cur_layer.activation_deriv_vectorized()
            dZdW = prev_layer.A

            delta = np.dot(delta, dZdA)

            dEdW = np.outer(delta, dZdW)
            dEdb = delta

            res.insert(0, (dEdW, dEdb))
            n -= 1
        return res


    def train(self, x: list[np.array], data: pd.DataFrame, labels: pd.DataFrame, batch_size: int=16, learning_rate: float=0.01):
        max = len(data.index)

        derivs = []
        
        for i in range(batch_size):
            row_num = np.random.randint(0, max)
            x = data.iloc[row_num]
            y = labels.iloc[row_num]
            self.feedforward(x)
            deriv = self.backprop_single(x, y)
            derivs.append(deriv)

        #accumulate all the gradients
        final_derivs = []

        for i in range(len(derivs[0])):
            dEdW_fin = ([deriv[i][0] for deriv in derivs].sum())/batch_size
            dEdb_fin = ([deriv[i][1] for deriv in derivs].sum())/batch_size
            final_derivs.append((dEdW_fin, dEdb_fin))

        #apply all the gradients
        for i in range(len(final_derivs)):
            dEdW = final_derivs[i][0]
            dEdb = final_derivs[i][1]
            self.layers[i].W -= learning_rate * dEdW
            self.layers[i].b -= learning_rate * dEdb
        
        
            

class sigmoid_network(network):
    
    def __init__(self, layers: list[int]):
        self.layers = [sigmoid_layer((layers[i], layers[i+1])) for i in range(len(layers) - 1)]

    

    